# Тест GPT adapter

Ноутбук проверяет, что `models/gpt.py` работает через общий интерфейс проекта: секреты берутся из `models/gpt/secrets/.env`, запуск идет через `runner.py`, результат пишется в `logs/`.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'runner.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
ROOT

## 1. Проверить секреты

Команда не печатает значение ключа, только проверяет, что `OPENAI_API_KEY` загружен.

In [ ]:
!python scripts/check_secrets.py --models gpt

## 2. Запустить через runner.py

Это основной CLI-интерфейс проекта.

In [ ]:
!python runner.py --problem data/competitions/local_examples/example.json --models gpt --run-id notebook_gpt_test

## 3. Посмотреть лог

Лог хранится в `logs/notebook_gpt_test.json`. Ниже выводятся только безопасные поля.

In [ ]:
import json

log = json.loads(Path('logs/notebook_gpt_test.json').read_text(encoding='utf-8'))
for result in log['results']:
    print('model:', result['model'])
    print('status:', 'error' if result['error'] else 'ok')
    print('tokens:', result['prompt_tokens'] + result['completion_tokens'])
    print('cost_usd:', result['cost_usd'])
    print('answer preview:', result['answer'][:500])
    print('error:', result['error'])

## 4. Прямой вызов адаптера

Это проверка интерфейса `BaseModel.solve(problem) -> SolveResult` без CLI.

In [ ]:
from runner import load_env
from models.gpt import GPTModel

load_env()
model = GPTModel()
result = model.solve('Докажите, что сумма двух четных чисел четна. Ответ дайте кратко.')
print(result.model, result.error, result.prompt_tokens + result.completion_tokens)
print(result.answer[:500])